# CIFAR-10 CNN with PCA-based First-Layer Filters
This notebook trains a CNN on CIFAR-10, performs PCA on the first convolutional layer filters, visualizes PCA score distributions with KDE, and reconstructs a new first layer using the top 64 PCA eigenvectors as filters.


In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
from sklearn.decomposition import PCA
import seaborn as sns
import matplotlib.pyplot as plt

# Set seeds and device
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


## Load CIFAR-10 dataset and create DataLoaders


In [ ]:
batch_size = 128
num_workers = 4

transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])

train_dataset = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=transform_train)
val_dataset = torchvision.datasets.CIFAR10(root="./data", train=True, download=False, transform=transform_test)
test_dataset = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=transform_test)

val_size = 5000
train_size = len(train_dataset) - val_size
train_dataset, _ = torch.utils.data.random_split(train_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(seed))
val_dataset = torch.utils.data.Subset(val_dataset, list(range(train_size, train_size + val_size)))

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)

print("Train samples:", len(train_dataset), "Validation samples:", len(val_dataset), "Test samples:", len(test_dataset))


## Define CNN model with two Conv layers of 128 filters


In [ ]:
class ConvNet(nn.Module):
    def __init__(self, first_out=128, second_out=128, num_classes=10):
        super(ConvNet, self).__init__()
        self.conv1 = nn.Conv2d(3, first_out, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(first_out)
        self.conv2 = nn.Conv2d(first_out, second_out, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(second_out)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc = nn.Linear(second_out * 8 * 8, num_classes)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

model = ConvNet(first_out=128, second_out=128).to(device)
print(model)


## Training loop: train & validate the CNN


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

train_history = []
val_history = []

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    return running_loss / total, correct / total

def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    return running_loss / total, correct / total

num_epochs = 5
checkpoint_path = "./cnn_cifar10_checkpoint.pth"
for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    scheduler.step()
    train_history.append((train_loss, train_acc))
    val_history.append((val_loss, val_acc))
    print(f"Epoch {epoch + 1}/{num_epochs}: train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, val_loss={val_loss:.4f}, val_acc={val_acc:.4f}")

torch.save({"model_state": model.state_dict(), "optimizer_state": optimizer.state_dict()}, checkpoint_path)
print("Saved checkpoint to", checkpoint_path)


## Extract first-layer filters and vectorize them


In [ ]:
checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint["model_state"])
model.eval()

conv1_weights = model.conv1.weight.data.cpu().numpy()  # shape [128, 3, 3, 3]
print("conv1 weights shape:", conv1_weights.shape)

num_filters, in_channels, kernel_h, kernel_w = conv1_weights.shape
X = conv1_weights.reshape(num_filters, -1)  # [128, 27]
print("Vectorized filter matrix shape:", X.shape)


## Compute PCA on filter vectors


In [ ]:
X_centered = X - X.mean(axis=0)
pca = PCA(n_components=min(X_centered.shape[0], X_centered.shape[1]))
pca.fit(X_centered)
components = pca.components_  # shape [n_components, 27]
explained_variance = pca.explained_variance_
explained_variance_ratio = pca.explained_variance_ratio_
print("PCA components shape:", components.shape)
print("Explained variance ratio (first 10):", explained_variance_ratio[:10])


## KDE visualization of PCA projections / eigenvector-score distributions


In [ ]:
projected = pca.transform(X_centered)  # shape [128, n_components]

plt.figure(figsize=(10, 4))
plt.plot(np.cumsum(explained_variance_ratio), marker="o")
plt.title("Cumulative Explained Variance by PCA Components")
plt.xlabel("Number of Components")
plt.ylabel("Cumulative Explained Variance")
plt.grid(True)
plt.show()

num_plot = min(8, projected.shape[1])
plt.figure(figsize=(14, 8))
for i in range(num_plot):
    plt.subplot(2, 4, i + 1)
    sns.kdeplot(projected[:, i], fill=True)
    plt.title(f"PC{i+1} score KDE")
    plt.xlabel("score")
    plt.tight_layout()
plt.show()


## Build new first-layer using top-64 eigenvectors as filters


In [ ]:
top_k = 64
selected_components = components[:top_k]
print("Selected components shape:", selected_components.shape)

new_filters = selected_components.reshape(top_k, in_channels, kernel_h, kernel_w)
new_filters_norm = new_filters / np.linalg.norm(new_filters.reshape(top_k, -1), axis=1, keepdims=True)
print("Reconstructed new filter bank shape:", new_filters_norm.shape)


## Replace the first layer with PCA-based filters and evaluate


In [ ]:
model_pca = ConvNet(first_out=top_k, second_out=128).to(device)
with torch.no_grad():
    model_pca.conv1.weight.copy_(torch.tensor(new_filters_norm, dtype=torch.float32))
    model_pca.conv2.weight.copy_(model.conv2.weight.data[:, :top_k, :, :])
    model_pca.bn1.weight.fill_(1.0)
    model_pca.bn1.bias.zero_()
    model_pca.bn2.weight.copy_(model.bn2.weight.data)
    model_pca.bn2.bias.copy_(model.bn2.bias.data)

val_loss_pca, val_acc_pca = validate(model_pca, val_loader, criterion, device)
print(f"PCA filter model val_loss={val_loss_pca:.4f}, val_acc={val_acc_pca:.4f}")

test_loss_pca, test_acc_pca = validate(model_pca, test_loader, criterion, device)
print(f"PCA filter model test_loss={test_loss_pca:.4f}, test_acc={test_acc_pca:.4f}")


### Optional fine-tuning of the PCA-based model


In [ ]:
finetune_epochs = 3
optimizer_pca = optim.SGD(model_pca.parameters(), lr=0.005, momentum=0.9, weight_decay=5e-4)
scheduler_pca = optim.lr_scheduler.StepLR(optimizer_pca, step_size=2, gamma=0.5)

for epoch in range(finetune_epochs):
    train_loss, train_acc = train_one_epoch(model_pca, train_loader, optimizer_pca, criterion, device)
    val_loss, val_acc = validate(model_pca, val_loader, criterion, device)
    scheduler_pca.step()
    print(f"Fine-tune epoch {epoch + 1}/{finetune_epochs}: train_loss={train_loss:.4f}, train_acc={train_acc:.4f}, val_loss={val_loss:.4f}, val_acc={val_acc:.4f}")

final_test_loss, final_test_acc = validate(model_pca, test_loader, criterion, device)
print(f"Final PCA-based model test_loss={final_test_loss:.4f}, test_acc={final_test_acc:.4f}")


## Save / load PCA filters and model checkpoints


In [ ]:
os.makedirs("saved_artifacts", exist_ok=True)
np.save("saved_artifacts/pca_components.npy", components)
np.save("saved_artifacts/pca_explained_variance.npy", explained_variance)
np.save("saved_artifacts/pca_explained_variance_ratio.npy", explained_variance_ratio)
np.save("saved_artifacts/pca_top64_filters.npy", new_filters_norm)
torch.save({"model_state": model_pca.state_dict()}, "saved_artifacts/model_pca_checkpoint.pth")
print("Saved PCA artifacts and PCA-based model checkpoint.")

# Example reload code:
# loaded_filters = np.load("saved_artifacts/pca_top64_filters.npy")
# pca_conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1, bias=False)
# with torch.no_grad():
#     pca_conv1.weight.copy_(torch.tensor(loaded_filters, dtype=torch.float32))
